# AB test 

A/B testing is a research method that allows you to find out people's reaction to any changes. The study shows which of the two versions of the product or offer is better and gives greater effect.

In [1]:
from hypex import HomogeneityTest
from hypex.dataset import Dataset, InfoRole, TargetRole, TreatmentRole
from hypex.utils import BackendsEnum, create_test_data

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


## Creation of a new test dataset with synthetic data.
It is important to mark the data fields by assigning the appropriate roles:

* FeatureRole: a role for columns that contain features or predictor variables. Our split will be based on them. Applied by default if the role is not specified for the column.
* TreatmentRole: a role for columns that show the treatment or intervention.
* TargetRole: a role for columns that show the target or outcome variable.
* InfoRole: a role for columns that contain information about the data, such as user IDs.

In [2]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "treat": TreatmentRole(),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": TargetRole(),
    },
    data=create_test_data(),
    backend=BackendsEnum.pandas,
)
data

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
0,0.0,7.0,1.0,489.0,476.888889,68.0,F,Logistics
1,1.0,9.0,1.0,504.5,444.777778,67.0,M,E-commerce
2,2.0,0.0,0.0,504.0,416.555556,48.0,F,E-commerce
3,3.0,9.0,1.0,490.0,443.111111,38.0,F,E-commerce
4,4.0,10.0,1.0,485.0,457.555556,67.0,F,Logistics
...,...,...,...,...,...,...,...,...
9995,9995.0,10.0,1.0,454.5,440.444444,43.0,F,E-commerce
9996,9996.0,0.0,0.0,478.0,410.666667,29.0,M,Finance
9997,9997.0,0.0,0.0,466.5,415.444444,59.0,F,Logistics
9998,9998.0,7.0,1.0,511.5,482.0,45.0,F,Finance


In [3]:
data.roles

{'user_id': Info(<class 'int'>),
 'treat': Treatment(<class 'float'>),
 'pre_spends': Target(<class 'float'>),
 'post_spends': Target(<class 'float'>),
 'gender': Target(<class 'str'>),
 'signup_month': Default(<class 'float'>),
 'age': Default(<class 'float'>),
 'industry': Default(<class 'str'>)}

## Homogeneity Test  

In [4]:
test = HomogeneityTest()
result = test.execute(data)

In [5]:
result.summary

,feature,group,control mean,test mean,difference,difference %,TTest pass,TTest p-value,KSTest pass,KSTest p-value,Chi2Test pass,Chi2Test p-value
0,pre_spends,1.0,484.621310,489.474829,4.853519,1.001507,NOT OK,1.035484e-33,NOT OK,1.624570e-14,NaN,NaN
1,post_spends,1.0,420.012249,484.534604,64.522355,15.362017,NOT OK,0.000000e+00,NOT OK,0.000000e+00,NaN,NaN
2,gender,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NOT OK,0.008733


## Working with the Spark backend

Everything above ran on the default **Pandas** backend. HypEx can run the same homogeneity test on **Spark**: the API of the tests and of their results stays the same, only a few things change:

1. Create a `SparkSession` and pass it to `Dataset` via `session=`.
2. Select the backend explicitly with `backend=BackendsEnum.spark`. `data` can still be a `pandas.DataFrame` - it is converted to a Spark DataFrame.
3. `Dataset.data` is a `pyspark.sql.DataFrame`, so use Spark API (`groupBy(...).count().show()`) instead of pandas methods (`groupby(...).agg(...)`) when you inspect it directly.
4. The computation is lazy and distributed - Spark is worth it for large data, for small data like in this tutorial pandas is faster.

### Create a Spark session

In [ ]:
import os

from pyspark.sql import SparkSession

from hypex.utils import BackendsEnum

# Drop leftover Spark settings from the environment and stop an already running session
for key in list(os.environ.keys()):
    if "SPARK" in key or "JAVA_OPTS" in key:
        del os.environ[key]
active = SparkSession.getActiveSession()
if active:
    active.stop()

# local[*] runs Spark on all cores of this machine; for a real cluster pass its master URL
sp_s = (
    SparkSession.builder.master("local[*]")
    .appName("HypExTutorial")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "4")  # for small data, less than the default 200
    .getOrCreate()
)
sp_s.sparkContext.setLogLevel("WARN")
sp_s

In [3]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "treat": TreatmentRole(),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": TargetRole(),
    },
    data=create_test_data(),
    backend=BackendsEnum.spark,
    session=sp_s,
)
data

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
0,0.0,0.0,0.0,460.5,423.111111,23.0,F,Logistics
1,1.0,6.0,1.0,489.0,492.0,31.0,M,Logistics
2,2.0,0.0,0.0,490.0,420.111111,26.0,M,E-commerce
3,3.0,10.0,1.0,485.0,449.333333,21.0,F,Logistics
4,4.0,6.0,1.0,492.5,491.777778,41.0,M,E-commerce
...,...,...,...,...,...,...,...,...
9995,9995.0,9.0,1.0,485.0,453.666667,27.0,F,Finance
9996,9996.0,6.0,1.0,505.5,483.888889,50.0,F,E-commerce
9997,9997.0,0.0,0.0,488.5,412.777778,18.0,F,E-commerce
9998,9998.0,11.0,1.0,503.5,432.777778,58.0,M,E-commerce


In [5]:
test = HomogeneityTest()
result = test.execute(data)

/Users/danilsamsutdinov/HypEx/hypex/dataset/backends/pandas_backend.py:1019: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_data = pd.concat([self.data] + [d.data for d in other], axis=axis)


In [6]:
result.summary

,feature,group,control mean,test mean,difference,difference %,TTest pass,TTest p-value,KSTest pass,KSTest p-value,Chi2Test pass,Chi2Test p-value
0,pre_spends,1.0,484.864121,489.971870,5.107749,1.053439,NOT OK,7.145004e-38,NOT OK,8.817732e-17,NaN,NaN
1,pre_spends,nan,484.864121,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,post_spends,1.0,419.815251,483.590319,63.775069,15.191223,NOT OK,0.000000e+00,NOT OK,0.000000e+00,NaN,NaN
3,post_spends,nan,419.815251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,gender,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OK,2.33276
